# Held-out validation sample and provider audit — preparation

**Companion to `tssc_analysis.ipynb` (v1.1.1).** This notebook produces the two blind-coding
spreadsheets for held-out inter-rater validation of the dictionary-coded constructs, and the
spreadsheet for the provider query-stratum audit.

**Run this locally, next to your own `reviews_raw.csv`.** It reproduces the exact cleaning and
dictionary-coding logic of `tssc_analysis.ipynb` so the sample is drawn from the same coded
corpus used in the reported analysis — it does not redefine or duplicate the dictionaries
independently. Nothing here is redistributed publicly: review text and provider identity stay
on your machine, in files you keep private and share only with your two coders and the auditor.

**Outputs (in `validation_output/`):**

| File | Who sees it |
|---|---|
| `coder_A_blind.xlsx` | Coder A only |
| `coder_B_blind.xlsx` | Coder B only |
| `provider_audit_blind.xlsx` | Provider auditor only |
| `answer_key_PRIVATE.xlsx` | You only — never share with coders or auditor before scoring |

After both coders and the auditor return their filled files, run `validation_scoring.ipynb`
to compute agreement statistics.


## 1. Configuration

In [ ]:
import hashlib, os, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

RAW_CSV = Path(os.getenv("TSSC_RAW_CSV", "reviews_raw.csv"))
OUTDIR = Path(os.getenv("TSSC_VALIDATION_OUTDIR", "validation_output"))
OUTDIR.mkdir(parents=True, exist_ok=True)

# Held-out review sample
N_TARGET = 2404              # total reviews in the held-out sample (both coders code the SAME set)
TARGET_PER_CONSTRUCT = 150   # positives sampled per construct, capped by how many exist
SEED_SAMPLING = 20260817     # matches the seed used in tssc_analysis.ipynb
SEED_ORDER_A = 20260903      # independent shuffles so coders don't see the same row order
SEED_ORDER_B = 20260904

# Provider audit
N_PROVIDER_AUDIT = 399
SEED_ORDER_AUDIT = 20260905

print(f"Reading raw corpus from: {RAW_CSV.resolve()}")


## 2. Reproduce the analytical corpus\n\nIdentical to Sections 1–3 of `tssc_analysis.ipynb`: same cleaning, exclusions, frozen dictionaries and negation corrections, so `df` here is the same coded corpus the reported analysis used. If you already have `df` in memory from running `tssc_analysis.ipynb` in this kernel, you can skip to Section 3 and reuse it directly instead of re-running this cell.

In [ ]:
raw = pd.read_csv(RAW_CSV)
required = {"place_id", "country", "city", "segment", "author_name", "rating", "text", "time_desc", "lang"}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = raw.copy()
for col in ["place_id", "country", "city", "segment", "author_name", "time_desc", "lang"]:
    df[col] = df[col].astype(str).str.strip()

NON_LEXICAL_SYMBOLS = "".join(chr(codepoint) for codepoint in [
    0x1FAF6, 0x1FA77, 0x1F979, 0x1FA75, 0x1FAF0, 0x1FAE1, 0x1FA76,
    0x1FAB7, 0x1FAE0, 0x1FABB, 0x1FAF4, 0x1FAE3, 0x1FAE4, 0x1FAF1,
    0x1FAF2, 0x1F6DC, 0x1FAA9, 0x1FAE2, 0x1FAB8,
])
NON_LEXICAL_PATTERN = "[" + re.escape(NON_LEXICAL_SYMBOLS) + "]"

def clean_text(value):
    if not isinstance(value, str):
        return np.nan
    value = unicodedata.normalize("NFC", value)
    value = re.sub(r"\(Traduzido pelo Google\)|\(Original\)", " ", value)
    value = "".join(ch for ch in value if ch == "\n" or unicodedata.category(ch)[0] != "C")
    value = re.sub(NON_LEXICAL_PATTERN, "", value)
    value = re.sub(r"[ \t]+", " ", value)
    value = re.sub(r"\n{3,}", "\n\n", value).strip()
    return value or np.nan

df["text"] = df["text"].apply(clean_text)
ambiguous_ids = set(df.groupby("place_id")["segment"].nunique().loc[lambda s: s > 1].index)

df = df[df["text"].notna()].copy()
df = df.drop_duplicates(subset=["place_id", "author_name", "text"], keep="first").copy()
df["repeated_text"] = df.duplicated(subset=["text"], keep=False) & df["text"].str.len().gt(15)
df["ambiguous_config"] = df["place_id"].isin(ambiguous_ids)
df["segment"] = df["place_id"].map(
    df.groupby("place_id")["segment"].agg(lambda s: s.value_counts().index[0])
)
df = df[~df["ambiguous_config"]].copy()

PT_NUM = {"um": 1, "uma": 1, "dois": 2, "duas": 2}
def to_years(value):
    value = value.lower()
    if any(token in value for token in ("hora", "dia", "\u00faltima semana", "ultima semana")):
        return 0.0
    match = re.search(r"(\d+)", value)
    number = int(match.group(1)) if match else PT_NUM.get(value.split()[0], 1)
    if "semana" in value:
        return round(number * 7 / 365, 3)
    if "m\u00eas" in value or "mes" in value:
        return round(number / 12, 3)
    if "ano" in value:
        return float(number)
    return np.nan

df["negative"] = df["rating"].le(2).astype(int)
df["n_words"] = df["text"].str.split().str.len()
df["log_words"] = np.log1p(df["n_words"])
df["review_age"] = df["time_desc"].apply(to_years)
df["translated"] = np.where(
    df["country"].eq("Brazil"),
    df["lang"].eq("pt-BR").astype(int),
    (~df["lang"].isin(["pt"])).astype(int),
)
df["mass_market"] = df["segment"].eq("mass_market").astype(int)

print(f"Analytical corpus: {len(df):,} reviews; {df.place_id.nunique():,} providers")


In [ ]:
# Frozen dictionaries — identical to tssc_analysis.ipynb Section 3
STAGES = {
 "stage_booking"   : r"reserv(a|ei|amos|ado)|agend(ei|amos|ado)|booking|pagamento|paguei|cart[\u00e3a]o de cr[\u00e9e]dito|dep[\u00f3o]sito|contrat(ei|amos|ado)",
 "stage_transport" : r"motorista|[\u00f4o]nibus|van\b|traslado|transfer\b|micro-?[\u00f4o]nibus|ve[\u00edi]culo|aeroporto|nos buscou|buscar no hotel|jipe|4x4|lancha|barco",
 "stage_lodging"   : r"hotel|pousada|hospedagem|resort|hostel|acomoda[\u00e7c][\u00e3a]o|di[\u00e1a]rias",
 "stage_food"      : r"almo[\u00e7c]o|jantar|caf[\u00e9e] da manh[\u00e3a]|refei[\u00e7c][\u00e3a]o|restaurante|lanche|degusta[\u00e7c][\u00e3a]o|comida servida",
 "stage_delivery"  : r"guia|instrutor|professor|monitor|anfitri[\u00e3a]o|oficina|aula|passeio|tour\b|atividade|experi[\u00eae]ncia",
}

CONSTRUCTS = {
 "intermediation" : r"ag[\u00eae]nci|operadora|tour operator|travels?\b|intermedi|terceiriz|subcontrat|revend|pacote (tur[\u00edi]stico|fechado|completo)",
 "coordination"   : r"bem organizad|super organizad|tudo organizad|organiza[\u00e7c][\u00e3a]o (foi|impec|perfeita|excelente)|planejad|itiner[\u00e1a]rio|roteiro|coordena[\u00e7c]|log[\u00edi]stic|bem estruturad|correu tudo (bem|certo)|sem contratempo|pontualidade",
 "delay"          : r"atras(o|os|ou|ado|ada|aram)|demor(ou|ada|ado|amos)|tempo de espera|longa espera|horas? de espera|fila (enorme|imensa|gigante)|esperamos (mais de|por|quase)|aguardamos (mais de|por)|n[\u00e3a]o chegou no hor[\u00e1a]rio",
 "digital"        : r"whatsapp|site\b|website|aplicativo|\bapp\b|on-?line|e-?mail|instagram|plataforma|link\b|reserva pela internet|formul[\u00e1a]rio",
 "environmental"  : r"meio ambiente|ambiental|sustent[\u00e1a]vel|sustentabilidade|ecol[\u00f3o]gic|reciclag|lixo|polui[\u00e7c]|impacto ambiental|pegada de carbono|conserva[\u00e7c][\u00e3a]o da natureza|preserva[\u00e7c][\u00e3a]o (ambiental|da natureza)|org[\u00e2a]nic",
 "labour"         : r"equipe|funcion[\u00e1a]ri|colaborador|treinad|capacitad|\bstaff\b|profissionalismo|artes[\u00e3a]|comunidade local|moradores locais|gera[\u00e7c][\u00e3a]o de (renda|emprego)",
 "price"          : r"pre[\u00e7c]o|caro|barato|taxa (extra|escondida|adicional)|cobra(ram|n\u00e7a|do)|custo|valor (cobrado|pago)|abusiv|superfaturad|custo-?benef[\u00edi]cio",
}

DISRUPTION = (
    r"cancel(ou|aram|ado|ada|amento)|"
    r"(?:n[\u00e3a]o|nunca|sem)\W+(?:\w+\W+){0,2}?(?:apareceu|apareceram|veio|vieram|compareceu|cumpriu|cumpriram|honrou|entregou|entregaram|devolveu|devolveram|reembolsou|reembolsaram)|"
    r"nos deixou na m[\u00e3a]o|deixaram (?:a gente|n[\u00f3o]s) na m[\u00e3a]o|remarcaram sem|sem aviso pr[\u00e9e]vio|"
    r"golpe|fraude|enganad|estelionat|n[\u00e3a]o cumpriram o (?:combinado|prometido)")

RECOVERY = (
    r"resolve(u|ram)|resolvid|solucion(ou|aram)|corrigi(u|ram)|pediu desculpa|se desculp|"
    r"reembolsaram|devolveram o dinheiro|compensa(ram|\u00e7\u00e3o)|trocaram por|deram um jeito|"
    r"remarcaram (para|sem custo)|nos acomodaram|refizeram")

NO_DELAY = r"sem (?:nenhum )?(?:atraso|demora)|n[\u00e3a]o (?:houve|teve|tivemos|tiveram) (?:nenhum )?atraso|nenhum atraso|zero atraso"
BENIGN_CXL = r"pol[\u00edi]tica de cancelamento|cancelamento (gr[\u00e1a]tis|gratuito|flex[\u00edi]vel|sem custo)|pode(m)? cancelar|cancelei|cancelamos|cancelar com anteced"

low = df.text.str.lower()
def match(pat):
    return low.str.contains(pat, regex=True, na=False).astype(int)

for k, p in {**STAGES, **CONSTRUCTS}.items():
    df[k] = match(p)

df["disruption"] = match(DISRUPTION)
df["recovery_terms"] = match(RECOVERY)
df.loc[match(BENIGN_CXL).astype(bool)
       & ~low.str.contains(r"cancel(?:ou|aram)\b", regex=True, na=False), "disruption"] = 0
df.loc[match(NO_DELAY).astype(bool)
       & ~low.str.contains(r"atras(?:o|ou|ado|aram) (?:de|no|na|em) ", regex=True, na=False), "delay"] = 0
df["recovery"] = ((df.disruption == 1) & (df.recovery_terms == 1)).astype(int)
df["stage_breadth"] = df[list(STAGES)].sum(axis=1)

CONSTRUCT_ORDER = ["disruption", "intermediation", "coordination", "delay",
                    "digital", "price", "labour", "environmental"]
for c in CONSTRUCT_ORDER:
    print(f"{c:16s} n={int(df[c].sum()):>6,}  ({100*df[c].mean():.2f}%)")


## 3. Diversity dimensions and the sampling function\n\nThe held-out sample must be usable evidence, not just a convenient subset: it needs to reflect the corpus on country, language exposure, provider segment, sentiment class and review length \u2014 not just be \"enough positives.\" `proportional_group_sample` allocates a target N across every combination of these dimensions in proportion to how common that combination is in the pool it draws from (largest-remainder rounding to hit the target exactly), and caps how many reviews any single provider can contribute, so the sample isn't dominated by a handful of prolific providers. This replaces a plain `GroupShuffleSplit`, which optimises for one grouping variable at a time and cannot jointly balance several stratifying dimensions.

In [ ]:
df["sentiment_class"] = np.select(
    [df.rating.le(2), df.rating.eq(3)], ["negative", "neutral"], default="positive"
)
df["length_bucket"] = np.where(df.n_words.ge(df.n_words.median()), "long", "short")

STRATA_COLS = ["country", "mass_market", "sentiment_class", "translated", "length_bucket"]

def proportional_group_sample(pool, strata_cols, n_target, seed, max_per_provider_share=0.03, exclude_idx=None):
    """Allocate n_target across strata proportionally to the pool's own composition
    (largest-remainder rounding), capping reviews per provider to preserve diversity."""
    pool = pool.copy()
    if exclude_idx:
        pool = pool[~pool.index.isin(exclude_idx)]
    if len(pool) == 0 or n_target <= 0:
        return []
    n_target = min(n_target, len(pool))
    pool["_stratum"] = pool[strata_cols].astype(str).agg("|".join, axis=1)
    props = pool["_stratum"].value_counts(normalize=True)
    raw = props * n_target
    base = np.floor(raw).astype(int)
    remainder = (raw - base).sort_values(ascending=False)
    shortfall = int(n_target - base.sum())
    for s in remainder.index[:shortfall]:
        base[s] += 1

    rng_local = np.random.default_rng(seed)
    max_per_provider = max(1, int(np.ceil(n_target * max_per_provider_share)))
    provider_count = {}
    chosen = []
    strata_order = list(base.index)
    rng_local.shuffle(strata_order)
    for stratum in strata_order:
        quota = int(base[stratum])
        if quota <= 0:
            continue
        candidates = pool.index[pool["_stratum"] == stratum].tolist()
        rng_local.shuffle(candidates)
        taken = 0
        for idx in candidates:
            if taken >= quota:
                break
            pid = pool.at[idx, "place_id"]
            if provider_count.get(pid, 0) >= max_per_provider:
                continue
            chosen.append(idx)
            provider_count[pid] = provider_count.get(pid, 0) + 1
            taken += 1
    return chosen


## 4. Held-out review sample (n \u2248 2,404, stratified across 8 constructs and demographics)\n\n**Phase 1** guarantees measurement coverage: for each of the eight constructs, up to `TARGET_PER_CONSTRUCT` dictionary-positive reviews are drawn \u2014 diversity-stratified *within that construct's own positive pool* \u2014 capped by however many positives actually exist (rare constructs such as `environmental` will be exhausted before the cap).\n\n**Phase 2** fills the remaining budget from reviews that are dictionary-negative on all eight constructs, again diversity-stratified, giving the comparison group needed to estimate specificity. Both coders code the *same* final set, independently and blind to the dictionary's codes, so agreement can be scored pairwise and against the dictionary.

In [ ]:
import hashlib

selected_idx = []
coverage = []
for c in CONSTRUCT_ORDER:
    pos_pool = df[df[c] == 1]
    take = min(TARGET_PER_CONSTRUCT, len(pos_pool))
    chosen = proportional_group_sample(pos_pool, STRATA_COLS, take,
                                        seed=SEED_SAMPLING, exclude_idx=set(selected_idx))
    selected_idx.extend(chosen)
    coverage.append({"construct": c, "available_positive": int(len(pos_pool)), "sampled": int(len(chosen))})

selected_idx = list(dict.fromkeys(selected_idx))
remaining = max(0, N_TARGET - len(selected_idx))
negative_pool = df[df[CONSTRUCT_ORDER].sum(axis=1) == 0]
negative_chosen = proportional_group_sample(negative_pool, STRATA_COLS, remaining,
                                             seed=SEED_SAMPLING + 1, exclude_idx=set(selected_idx))
coverage.append({
    "construct": "ALL_NEGATIVE (specificity control)",
    "available_positive": int(len(negative_pool)), "sampled": int(len(negative_chosen)),
})

final_idx = list(dict.fromkeys(selected_idx + negative_chosen))
review_sample = df.loc[final_idx].copy()

review_sample["review_id"] = (
    review_sample["place_id"].astype(str) + "||" + review_sample["text"].astype(str)
).apply(lambda s: "R" + hashlib.sha256(s.encode("utf-8")).hexdigest()[:10])
review_sample["provider_pseudo"] = "P" + pd.Series(
    pd.factorize(review_sample["place_id"])[0], index=review_sample.index
).astype(str).str.zfill(5)

coverage_report = pd.DataFrame(coverage)
print(f"Held-out sample size: {len(review_sample):,} (target {N_TARGET:,})")
if len(review_sample) < N_TARGET:
    print("Below target: the negative pool ran out, or TARGET_PER_CONSTRUCT is too low relative "
          "to N_TARGET. Raise TARGET_PER_CONSTRUCT, or lower N_TARGET, and re-run.")
coverage_report


In [ ]:
print("--- Diversity check: held-out sample vs full corpus ---")
for col in STRATA_COLS:
    corpus_p = df[col].value_counts(normalize=True).round(3)
    sample_p = review_sample[col].value_counts(normalize=True).round(3)
    print(f"\n{col}")
    display(pd.DataFrame({"corpus": corpus_p, "sample": sample_p}).fillna(0))

print(f"\nUnique providers in sample: {review_sample.place_id.nunique()} "
      f"| max reviews from one provider: {review_sample.place_id.value_counts().max()}")


## 5. Blind coder spreadsheets\n\nEach coder sees only `review_text`, in an order independently shuffled per coder, with the eight construct columns left blank for 0/1 coding **and an independent `sentiment` column** (positive / neutral / negative), used later to check whether the star-rating-derived sentiment class the models rely on agrees with what an independent reader perceives in the text. No automated code, rating, or provider information is shown to the coder.

In [ ]:
CONSTRUCT_LABELS = {
    "disruption": "Service disruption \u2014 cancellation, no-show, unfulfilled promise, fraud/scam language.",
    "intermediation": "Booking mediated by an agency/tour operator/reseller rather than direct provider contact.",
    "coordination": "Explicit praise for organisation, itinerary planning, logistics or punctuality.",
    "delay": "Reported waiting time, lateness or late arrival.",
    "digital": "Mentions a digital channel: WhatsApp, website, app, email, online booking/platform.",
    "price": "Mentions price, cost, fees, value for money, or being overcharged.",
    "labour": "Mentions staff, employees, training, professionalism, or local workforce/community.",
    "environmental": "Mentions environment, sustainability, ecology, waste, pollution, or conservation.",
}

def make_coder_sheet(sample, seed):
    order = sample.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    sheet = pd.DataFrame({
        "validation_id": ["V" + str(i + 1).zfill(4) for i in range(len(order))],
        "review_text": order["text"].values,
    })
    for c in CONSTRUCT_ORDER:
        sheet[c] = ""
    sheet["sentiment"] = ""   # coder enters: positive / neutral / negative
    sheet["notes"] = ""
    key = pd.DataFrame({"validation_id": sheet["validation_id"], "review_id": order["review_id"].values})
    return sheet, key

coder_A, key_A = make_coder_sheet(review_sample, SEED_ORDER_A)
coder_B, key_B = make_coder_sheet(review_sample, SEED_ORDER_B)

instructions = pd.DataFrame({
    "column": ["review_text"] + CONSTRUCT_ORDER + ["sentiment", "notes"],
    "what to do": (
        ["Read the review text."]
        + [f"Mark 1 if present, 0 if absent. {CONSTRUCT_LABELS[c]}" for c in CONSTRUCT_ORDER]
        + ["Your independent judgement of overall tone: 'positive', 'neutral', or 'negative'. "
           "Judge from the text, not from any rating (none is shown)."]
        + ["Leave blank unless a case is genuinely ambiguous; if so, note why briefly."]
    ),
})

def write_coder_workbook(path, sheet):
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        instructions.to_excel(xw, sheet_name="Instructions", index=False)
        sheet.to_excel(xw, sheet_name="Coding", index=False)

write_coder_workbook(OUTDIR / "coder_A_blind.xlsx", coder_A)
write_coder_workbook(OUTDIR / "coder_B_blind.xlsx", coder_B)
print("Wrote", OUTDIR / "coder_A_blind.xlsx", "and", OUTDIR / "coder_B_blind.xlsx")


## 6. Provider query-stratum audit (n = 399)

**What this audit can and cannot establish.** The `mass_market` variable in `tssc_analysis.ipynb` is not a verified organizational characteristic of the provider: it records which retrieval query returned that provider. Throughout the validation materials it is therefore named a **query stratum** — `mass-market query stratum` vs `creative-tourism query stratum` — and results are read as patterns associated with the retrieval strategy, not as properties of the firms.

An earlier version of this audit asked an independent assessor to classify each provider blind, as 'mass_market' or 'creative', and compared that judgement against the automated label. On the real corpus that comparison returned chance-level agreement (Cohen's κ ≈ −0.01, n = 399): the assessor was being asked to recover an organizational typology that the label never encoded, from a workbook that at that point did not even identify the business. That negative result is reported as a limit of the proxy, and it is why the audit below asks a different, answerable question.

**The question asked now** is face validity of the retrieval: given the query stratum that returned this provider, is the business plausibly a hit for that query? The stratum is shown to the assessor rather than hidden, since it is the thing being checked. Providers are ranked by number of reported disruptions, then by review volume — the audit targets the providers where the stratum matters most for the reported contrast.

In [ ]:
STRATUM_LABELS = {1: "mass-market query stratum", 0: "creative-tourism query stratum"}

provider = (df.groupby("place_id")
            .agg(country=("country", "first"), city=("city", "first"),
                 mass_market=("mass_market", "first"),
                 n_reviews=("rating", "size"), mean_rating=("rating", "mean"),
                 disruption_n=("disruption", "sum"))
            .reset_index())
provider["disruption_n"] = provider["disruption_n"].astype(int)
# query_stratum is the canonical name in the validation materials; mass_market is kept as a
# compatibility alias so these outputs still join to tssc_analysis.ipynb, whose columns are unchanged.
provider["query_stratum"] = provider["mass_market"].astype(int).map(STRATUM_LABELS)
provider["provider_pseudo"] = "P" + pd.Series(
    pd.factorize(provider["place_id"])[0], index=provider.index
).astype(str).str.zfill(5)

priority = provider.sort_values(["disruption_n", "n_reviews"], ascending=[False, False])
audit_sample = priority.head(min(N_PROVIDER_AUDIT, len(priority))).copy()
print(f"Providers audited: {len(audit_sample)} (of {len(provider)} total); "
      f"with >=1 reported disruption: {int((audit_sample.disruption_n > 0).sum())}")

audit_sample["google_maps_link"] = (
    "https://www.google.com/maps/place/?q=place_id:" + audit_sample["place_id"].astype(str)
)

audit_blind = audit_sample[
    ["provider_pseudo", "google_maps_link", "query_stratum", "country", "city",
     "n_reviews", "mean_rating"]
].copy()
audit_blind = audit_blind.sample(frac=1.0, random_state=SEED_ORDER_AUDIT).reset_index(drop=True)
audit_blind["stratum_fit"] = ""      # assessor fills: consistent / inconsistent / undetermined
audit_blind["auditor_notes"] = ""

audit_instructions = pd.DataFrame({"note": [
    "This audit checks the retrieval, not the firm. Each provider was returned by one of two search "
    "strategies (its 'query_stratum'): a mass-market query stratum (standardized, large-group/package "
    "tourism searches) or a creative-tourism query stratum (boutique, small-scale, personalised or "
    "local-experience searches).",
    "For each provider, open 'google_maps_link' to see the actual business (name, photos, category, "
    "reviews on Google Maps) and judge whether it is plausibly a hit for the query stratum shown in "
    "'query_stratum'. Enter 'consistent' if a reasonable person running that kind of search would "
    "expect this business among the results, 'inconsistent' if it clearly belongs to the other "
    "stratum, or 'undetermined' if the listing carries too little information to tell. Use "
    "'auditor_notes' for anything ambiguous.",
    "You are NOT being asked to classify the business as mass-market or creative. That typology is "
    "not what the stratum encodes, and an earlier version of this audit that asked for it produced "
    "chance-level agreement. Judge only the fit between the business and the query stratum that "
    "retrieved it.",
    "The Google Place ID behind the link is a public Google identifier, not personal data \u2014 it "
    "exists only to let you look up the business, and stays out of every publicly shared file from "
    "this study (GitHub/Zenodo). Please do not forward this workbook outside the audit team.",
]})

with pd.ExcelWriter(OUTDIR / "provider_audit_blind.xlsx", engine="openpyxl") as xw:
    audit_instructions.to_excel(xw, sheet_name="Instructions", index=False)
    audit_blind.to_excel(xw, sheet_name="Audit", index=False)
print("Wrote", OUTDIR / "provider_audit_blind.xlsx")

audit_key = audit_sample[
    ["provider_pseudo", "place_id", "query_stratum", "mass_market", "disruption_n"]
].copy()


## 7. Private answer key\n\n**Keep this file to yourself.** It links `validation_id` / `provider_pseudo` back to the dictionary's codes, the rating-derived sentiment class, and the query stratum, needed by `validation_scoring.ipynb`. Never share it with the coders or the auditor before scoring.

In [ ]:
review_answer_key = review_sample[
    ["review_id"] + CONSTRUCT_ORDER
    + ["sentiment_class", "place_id", "provider_pseudo",
       "country", "city", "mass_market", "translated", "n_words", "length_bucket"]
].copy()

with pd.ExcelWriter(OUTDIR / "answer_key_PRIVATE.xlsx", engine="openpyxl") as xw:
    key_A.merge(review_answer_key, on="review_id").to_excel(xw, sheet_name="coder_A_key", index=False)
    key_B.merge(review_answer_key, on="review_id").to_excel(xw, sheet_name="coder_B_key", index=False)
    audit_key.to_excel(xw, sheet_name="provider_audit_key", index=False)
    coverage_report.to_excel(xw, sheet_name="sampling_coverage", index=False)

print("Wrote", OUTDIR / "answer_key_PRIVATE.xlsx")
print("\nDone. Files in", OUTDIR.resolve())
for f in sorted(OUTDIR.iterdir()):
    print(" -", f.name)
